# AstroSpectro · PHY-3500 — Notebook 03 : Autoencodeur

**Auteurs** : Alex, Justine  
**Cours** : PHY-3500 Physique numérique — Université Laval  
**Prérequis** : Notebook 01 (`01_pca.ipynb`) — features + PCAAnalyzer requis.

---

## Objectifs

1. Entraîner un **autoencodeur MLP** sur les features spectrales LAMOST DR5.
2. Visualiser et interpréter l'**espace latent 2D** (comparer à PCA et UMAP).
3. Évaluer la **qualité de reconstruction** par classe et comparer avec PCA.
4. Démontrer la **continuité de l'espace latent** via interpolation.
5. Conclure sur les forces et limites de chaque méthode (PCA / UMAP / AE).

---

## Plan

- §0 · Imports & configuration  
- §1 · Chargement des données (features V2)  
- §2 · Architecture & entraînement de l'autoencodeur  
- §3 · Espace latent 2D — visualisation et interprétation physique  
- §4 · Qualité de reconstruction — autoencodeur vs PCA  
- §5 · Continuité de l'espace latent — interpolation  
- §6 · Synthèse comparative PCA / UMAP / AE

## 0 · Imports & configuration

In [ ]:
import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

# ── Environnement projet ───────────────────────────────────────────────────
sys.path.insert(0, str(Path("../..").resolve() / "src"))

from utils import setup_project_env, latest_file

paths = setup_project_env(verbose=True)

# ── Chemins ────────────────────────────────────────────────────────────────
FEATURES_PATH = latest_file(paths["PROCESSED_DIR"], "features_*.csv")
if FEATURES_PATH is None:
    raise FileNotFoundError(
        "Aucun fichier features_*.csv trouvé dans data/reports/\n"
        "→ Lancer d'abord 00_master_pipeline.ipynb pour générer les features."
    )
FEATURES_PATH  = Path(FEATURES_PATH)
CATALOG_PATH   = Path(paths["CATALOG_DIR"]) / "master_catalog_gaia.csv"
FIGURES_ROOT   = Path(paths["REPORTS_DIR"]) / "figures" / "phy3500"
FIGURES_DIR    = FIGURES_ROOT / FEATURES_PATH.stem
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR     = Path(paths["REPORTS_DIR"]) / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nFeatures  : {FEATURES_PATH}")
print(f"Catalog   : {CATALOG_PATH}")
print(f"Figures   : {FIGURES_DIR}")
print(f"Models    : {MODELS_DIR}")

# ── Imports dimred ─────────────────────────────────────────────────────────
from dimred import DimRedDataLoader, PCAAnalyzer, DimRedVisualizer
from dimred.autoencoder import SpectralAutoencoder

# ── Logging & reproductibilité ─────────────────────────────────────────────
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(name)s | %(message)s")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("\n✓ Environnement prêt.")

## 1 · Chargement des données

On réutilise exactement le même pipeline de chargement que les notebooks 01 et 02
pour garantir la cohérence des résultats.

In [ ]:
loader = DimRedDataLoader(
    features_path=FEATURES_PATH,
    catalog_path=CATALOG_PATH,
    random_state=RANDOM_STATE,
)

X, y, meta = loader.load(
    mode="features",
    snr_min=10.0,
    scale=True,          # StandardScaler — obligatoire pour l'AE
    class_balance=False,
)

feature_names = loader.get_feature_names()
input_dim     = X.shape[1]

print(f"\nDonnées chargées :")
print(f"  X         : {X.shape}")
print(f"  y         : {y.shape} | classes = {dict(zip(*np.unique(y, return_counts=True)))}")
print(f"  meta      : {meta.shape}")
print(f"  input_dim : {input_dim}")

In [ ]:
# Chargement du PCAAnalyzer sauvegardé (depuis notebook 01)
# ── Cherche le modèle PCA dans MODELS_DIR ─────────────────────────────────
pca_model_path = MODELS_DIR / "pca_analyzer.joblib"

if pca_model_path.exists():
    pca = PCAAnalyzer.load(str(pca_model_path))
    scores_pca = pca.transform(X)
    print(f"✓ PCAAnalyzer chargé | {pca.sklearn_pca.n_components_} composantes")
    print(f"  scores_pca : {scores_pca.shape}")
else:
    # Réentraîne PCA si modèle absent
    print("⚠  PCAAnalyzer non trouvé → réentraînement...")
    pca = PCAAnalyzer(n_components=50, random_state=RANDOM_STATE)
    scores_pca = pca.fit_transform(X, feature_names=feature_names)
    pca.save(str(pca_model_path))
    print(f"✓ PCAAnalyzer entraîné et sauvegardé")

n_pcs_95 = pca.n_components_for_variance(0.95)
print(f"  95% variance → {n_pcs_95} composantes PCA")

## 2 · Architecture & entraînement de l'autoencodeur

### Architecture choisie

```
X (D) → [256] → [128] → [64] → z (2) → [64] → [128] → [256] → X̂ (D)
```

- **Activations** : ReLU (couches cachées) + BatchNorm + Dropout(0.1)  
- **Sortie** : activation linéaire (les features sont déjà standardisées)  
- **Loss** : MSE — cohérent avec la PCA (minimisation de l'erreur de reconstruction)  
- **Optimiseur** : Adam + ReduceLROnPlateau + early stopping

### Choix de `latent_dim = 2`

On force 2D pour permettre la **comparaison directe** avec PCA (PC1/PC2)  
et UMAP. C'est un choix délibérément contraignant — l'idée est de voir  
si le réseau peut capturer la même structure physique que PCA/UMAP  
dans une représentation non-linéaire à 2 dimensions.

In [ ]:
# ── Construction du modèle ────────────────────────────────────────────────
ae = SpectralAutoencoder(
    input_dim=input_dim,
    latent_dim=2,               # 2D pour comparaison directe PCA / UMAP
    hidden_dims=[256, 128, 64],
    dropout=0.1,
)

print(ae)
print(f"\nDevice : {ae.device}")

# Compte des paramètres (info pour le rapport)
try:
    import torch
    n_params = sum(p.numel() for p in ae._model.parameters()
                   if p is not None and p.requires_grad)
    # Reconstruit _model pour le comptage avant fit()
except Exception:
    pass

In [ ]:
# ── Entraînement ─────────────────────────────────────────────────────────
# Vérifie si un modèle sauvegardé existe déjà
ae_model_path = MODELS_DIR / "ae_latent2.pt"

if ae_model_path.exists():
    print("✓ Modèle autoencodeur trouvé — chargement...")
    ae = SpectralAutoencoder.load(str(ae_model_path))
    history = ae.history_
    print(f"  fit_time : {ae.fit_time_:.1f}s")
    print(f"  best val_loss : {min(history['val_loss']):.6f}")
else:
    print("Entraînement de l'autoencodeur (latent_dim=2)...")
    history = ae.fit(
        X,
        epochs=200,
        lr=1e-3,
        batch_size=512,
        val_fraction=0.10,
        weight_decay=1e-5,
        lr_scheduler=True,
        early_stopping_patience=25,
        verbose=True,
    )
    ae.save(str(ae_model_path))
    print(f"\n✓ Modèle sauvegardé : {ae_model_path}")

print(f"\nEpochs effectuées : {len(history['train_loss'])}")
print(f"Best val_loss     : {min(history['val_loss']):.6f}")

In [ ]:
# ── Courbe d'apprentissage ────────────────────────────────────────────────
viz = DimRedVisualizer(figsize=(12, 4), dpi=150, output_dir=FIGURES_DIR)

fig, axes = viz.plot_training_history(
    history,
    save_path=FIGURES_DIR / "ae_training_history.png",
)
plt.show()

## 3 · Espace latent 2D — visualisation et interprétation physique

On encode l'ensemble du jeu de données dans l'espace latent de dimension 2,
puis on colore par classe, T_eff, log g, [Fe/H] et G_BP-G_RP —
exactement comme pour PCA (§3 notebook 01) et UMAP (§1 notebook 02).

In [ ]:
# ── Encodage ──────────────────────────────────────────────────────────────
Z_ae = ae.encode(X)
print(f"Espace latent : {Z_ae.shape}")
print(f"  axe 1 : [{Z_ae[:,0].min():.2f}, {Z_ae[:,0].max():.2f}]")
print(f"  axe 2 : [{Z_ae[:,1].min():.2f}, {Z_ae[:,1].max():.2f}]")

In [ ]:
# ── Grille de visualisations ──────────────────────────────────────────────
fig, axes = viz.plot_ae_embedding(
    Z_ae, meta, y,
    save_path=FIGURES_DIR / "ae_latent_grid.png",
)
plt.show()

### Corrélations espace latent ↔ paramètres physiques (Spearman)

On calcule les corrélations entre les 2 axes latents et T_eff, log g, [Fe/H]
pour quantifier quelle physique l'AE a encodé — et comparer avec PC1/PC2.

In [ ]:
# Corrélations espace latent vs paramètres physiques
from scipy.stats import spearmanr

phys_cols = ["teff_gspphot", "logg_gspphot", "mh_gspphot", "bp_rp"]
phys_labels = ["T_eff", "log g", "[Fe/H]", "G_BP-G_RP"]

print("Corrélations Spearman : Espace latent AE ↔ Paramètres physiques")
print("-" * 60)
print(f"{'Paramètre':<15} {'Axe latent 1':>14} {'Axe latent 2':>14}")
print("-" * 60)

for col, lbl in zip(phys_cols, phys_labels):
    if col not in meta.columns:
        continue
    vals = meta[col].values.astype(float)
    valid = np.isfinite(vals)
    r1, _ = spearmanr(Z_ae[valid, 0], vals[valid])
    r2, _ = spearmanr(Z_ae[valid, 1], vals[valid])
    print(f"{lbl:<15} {r1:>+14.3f} {r2:>+14.3f}")

print("-" * 60)

# Comparaison directe avec PC1/PC2
print("\nRappel — PC1/PC2 (notebook 01) :")
print(f"{'Paramètre':<15} {'PC1':>14} {'PC2':>14}")
print("-" * 60)
corr_pca = pca.correlations_with_params(meta[phys_cols], scores=scores_pca, n_pcs=2)
for col, lbl in zip(phys_cols, phys_labels):
    if col not in corr_pca.columns:
        continue
    print(f"{lbl:<15} {corr_pca.loc['PC1', col]:>+14.3f} {corr_pca.loc['PC2', col]:>+14.3f}")

## 4 · Qualité de reconstruction — autoencodeur vs PCA

### 4.1 Exemples de reconstruction

In [ ]:
# ── Reconstruction de l'ensemble ─────────────────────────────────────────
X_recon = ae.reconstruct(X)

print(f"MSE globale autoencodeur (latent_dim=2) : {ae.reconstruction_mse(X):.5f}")
print(f"MSE globale PCA (n=2)                   : "
      f"{pca.reconstruction_error(X, n_components=2).mean():.5f}")
print(f"MSE globale PCA (n={n_pcs_95})                  : "
      f"{pca.reconstruction_error(X, n_components=n_pcs_95).mean():.5f}")

In [ ]:
# ── Exemples visuels : original vs reconstruit ────────────────────────────
fig, axes = viz.plot_ae_reconstruction(
    X, X_recon, feature_names,
    n_samples=5,
    y=y,
    save_path=FIGURES_DIR / "ae_reconstruction_examples.png",
)
plt.show()

### 4.2 Comparaison MSE autoencodeur vs PCA

In [ ]:
# ── Comparaison AE vs PCA ─────────────────────────────────────────────────
comparison_df = ae.compare_with_pca(
    X, pca,
    n_pcs_list=[1, 2, 3, 5, 8, 10, 15, 20, 30, 50],
)
print(comparison_df.to_string(index=False))

fig, ax = viz.plot_ae_vs_pca(
    comparison_df,
    save_path=FIGURES_DIR / "ae_vs_pca_mse.png",
)
plt.show()

### 4.3 Distribution des erreurs par classe

In [ ]:
# ── Résumé par classe ─────────────────────────────────────────────────────
summary = ae.reconstruction_summary(X, y=y)
print("\nErreurs de reconstruction par classe :")
print(summary.to_string(index=False))

# Distribution
mse_per = ae.reconstruction_mse(X, per_sample=True)

fig, ax = viz.plot_ae_error_distribution(
    mse_per, y,
    save_path=FIGURES_DIR / "ae_error_distribution.png",
)
plt.show()

## 5 · Continuité de l'espace latent — interpolation

Une propriété clé d'un bon espace latent est sa **continuité** :
interpoler linéairement entre deux points dans l'espace latent doit
produire des reconstructions physiquement cohérentes.

Ici on interpole entre une **étoile froide** (T_eff ~ 4000 K)
et une **étoile chaude** (T_eff ~ 8000 K) et on observe comment
les features reconstruites évoluent.

In [ ]:
# Sélection d'une étoile froide et d'une étoile chaude
teff = meta["teff_gspphot"].values
valid_teff = np.isfinite(teff)

# Étoile froide : T_eff la plus basse dans le top-SNR
cold_candidates = np.where(valid_teff & (teff < 4200))[0]
hot_candidates  = np.where(valid_teff & (teff > 7500))[0]

if len(cold_candidates) == 0 or len(hot_candidates) == 0:
    print("⚠ Pas assez de candidats — utilisation des percentiles 5 et 95")
    cold_candidates = np.where(valid_teff & (teff < np.nanpercentile(teff, 5)))[0]
    hot_candidates  = np.where(valid_teff & (teff > np.nanpercentile(teff, 95)))[0]

# Prend le meilleur SNR dans chaque groupe
snr_r = meta["snr_r"].values if "snr_r" in meta.columns else np.ones(len(X))

idx_cold = cold_candidates[np.argmax(snr_r[cold_candidates])]
idx_hot  = hot_candidates[np.argmax(snr_r[hot_candidates])]

teff_cold = teff[idx_cold]
teff_hot  = teff[idx_hot]
label_a   = f"Froide (T_eff={teff_cold:.0f} K)"
label_b   = f"Chaude (T_eff={teff_hot:.0f} K)"

print(f"Étoile A (froide) : indice {idx_cold}, T_eff={teff_cold:.0f} K")
print(f"Étoile B (chaude) : indice {idx_hot},  T_eff={teff_hot:.0f} K")

In [ ]:
# ── Interpolation dans l'espace latent ────────────────────────────────────
Z_interp, X_interp = ae.latent_interpolation(
    X[idx_cold], X[idx_hot],
    n_steps=15,
)

fig, axes = viz.plot_latent_interpolation(
    Z_interp, X_interp,
    feature_names=feature_names,
    label_a=label_a,
    label_b=label_b,
    save_path=FIGURES_DIR / "ae_latent_interpolation.png",
)
plt.show()

print("\nTrajectoire latente :")
print(f"  Départ  : z = ({Z_interp[0,0]:.3f}, {Z_interp[0,1]:.3f})")
print(f"  Arrivée : z = ({Z_interp[-1,0]:.3f}, {Z_interp[-1,1]:.3f})")
print(f"  Distance euclidienne dans l'espace latent : "
      f"{np.linalg.norm(Z_interp[-1] - Z_interp[0]):.3f}")

## 6 · Synthèse comparative PCA / UMAP / Autoencodeur

In [ ]:
# ── Tableau comparatif quantitatif ────────────────────────────────────────
from scipy.stats import spearmanr

teff_valid = np.isfinite(meta["teff_gspphot"].values)
teff_vals  = meta["teff_gspphot"].values[teff_valid]

# Chargement des embeddings UMAP (sauvegardés notebook 02)
umap_path = FIGURES_DIR.parent / "Z_umap.npy"
Z_umap = np.load(umap_path) if umap_path.exists() else None

results = {}

# PCA
r_pca, _ = spearmanr(scores_pca[teff_valid, 0], teff_vals)
results["PCA (PC1)"] = {
    "Corrélation T_eff (axe 1)": f"{r_pca:+.3f}",
    "MSE reconstruction (2 dims)": f"{pca.reconstruction_error(X, n_components=2).mean():.5f}",
    "Paramétrique": "Oui",
    "Non-linéaire": "Non",
    "Interprétable": "Oui",
}

# Autoencodeur
r_ae, _ = spearmanr(Z_ae[teff_valid, 0], teff_vals)
results["AE (latent 1)"] = {
    "Corrélation T_eff (axe 1)": f"{r_ae:+.3f}",
    "MSE reconstruction (2 dims)": f"{ae.reconstruction_mse(X):.5f}",
    "Paramétrique": "Oui",
    "Non-linéaire": "Oui",
    "Interprétable": "Partielle",
}

# UMAP (si disponible)
if Z_umap is not None:
    r_umap, _ = spearmanr(Z_umap[teff_valid, 0], teff_vals)
    results["UMAP (axe 1)"] = {
        "Corrélation T_eff (axe 1)": f"{r_umap:+.3f}",
        "MSE reconstruction (2 dims)": "N/A",
        "Paramétrique": "Oui (partiel)",
        "Non-linéaire": "Oui",
        "Interprétable": "Non",
    }

df_summary = pd.DataFrame(results).T
print("\n=== Synthèse comparative ===")
print(df_summary.to_string())

## Conclusions

### Ce que l'autoencodeur apporte vs PCA

| Critère | PCA | Autoencodeur |
|---------|-----|--------------|
| Non-linéarité | ✗ | ✓ |
| Paramétrique (transform nouveaux) | ✓ | ✓ |
| MSE reconstruction (2 dims) | *à compléter* | *à compléter* |
| Interprétabilité des axes | ✓ (loadings) | ✗ (black-box) |
| Temps d'entraînement | < 1 s | *à compléter* |
| Stabilité | ✓ | Variable selon init |

### Limitations observées

1. **Espace latent 2D très contraint** : latent_dim=2 est une compression  
   extrême — la MSE sera forcément supérieure à PCA avec n_pcs=5.

2. **Non-interprétabilité des axes** : contrairement à PC1 (température)  
   et PC2 (métallicité), les axes latents AE n'ont pas d'interprétation  
   physique directe sans analyse supplémentaire.

3. **Sensibilité à l'initialisation** : l'autoencodeur peut converger vers  
   des minima locaux différents selon la seed et l'architecture.

### Recommandation pour AstroSpectro pipeline principal

Pour la **classification XGBoost**, les **scores PCA** (ou les features  
physiques V2 directement) restent préférables : plus interprétables,  
déterministes, et prouvés efficaces à 84%+. L'autoencodeur est plus  
pertinent pour l'**exploration non-supervisée** et la détection d'anomalies.